# Prophet Model — Experiment Notebook
**Task 3: Time Series Forecasting (Facebook Prophet)**

This notebook:
1. Loads preprocessed data for all stocks
2. Fits Prophet with customisable seasonality
3. Evaluates on test set (Jul–Dec 2025)
4. Forecasts next 5 trading days with confidence intervals
5. Computes RMSE, MAPE, Directional Accuracy

In [ ]:
import sys
sys.path.append("..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet

from src.data.fetch_data import load_all_raw, STOCK_UNIVERSE
from src.data.preprocess import preprocess_all
from src.utils.metrics import evaluate
from src.models.prophet_model import (
    series_to_prophet_df, fit_prophet,
    predict_test_prophet, forecast_future_prophet,
    run_prophet_pipeline
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
print("Imports OK")

## 1. Load & Preprocess Data

In [ ]:
raw = load_all_raw()
processed = preprocess_all(raw, save=False)
print(f"Loaded {len(processed)} stocks")

## 2. Single-Stock Deep Dive (TCS.NS)

In [ ]:
TICKER = "TCS.NS"
train = processed[TICKER]["train"]
test  = processed[TICKER]["test"]

# Fit Prophet
model = fit_prophet(train, changepoint_prior_scale=0.05)

# Predict test period
pred = predict_test_prophet(model, test)

# Evaluate
metrics = evaluate(test.values, pred.values, "Prophet", TICKER)
print(f"Metrics for {TICKER}:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# Plot forecast vs actual
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train[-100:], label="Train (last 100d)", color="steelblue", alpha=0.5)
ax.plot(test, label="Actual", color="green", linewidth=2)
ax.plot(pred, label="Prophet", color="orange", linewidth=1.5, linestyle="--")
ax.set_title(f"{TICKER} — Prophet Forecast vs Actual")
ax.legend()
plt.tight_layout()
plt.show()

### Prophet Components

In [ ]:
# Prophet component plots (trend, weekly, yearly seasonality)
train_df = series_to_prophet_df(train)
m2 = Prophet(daily_seasonality=True, weekly_seasonality=True,
             yearly_seasonality=True, interval_width=0.95)
m2.fit(train_df)
future = m2.make_future_dataframe(periods=len(test) + 5, freq="B")
fc_full = m2.predict(future)

fig = m2.plot_components(fc_full)
plt.suptitle(f"{TICKER} — Prophet Components", y=1.02)
plt.tight_layout()
plt.show()

### Future Forecast with Confidence Intervals

In [ ]:
fc_vals, fc_lo, fc_hi = forecast_future_prophet(model, train, test, n_periods=5)
full = pd.concat([train, test])
future_dates = pd.bdate_range(start=full.index[-1] + pd.Timedelta(days=1), periods=5)

print(f"\nProphet Forecast — {TICKER} — Next 5 Trading Days:")
for d, v, lo, hi in zip(future_dates, fc_vals, fc_lo, fc_hi):
    print(f"  {d.date()}: ₹{v:.2f}  [₹{lo:.2f} — ₹{hi:.2f}]")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(full[-60:], label="Historical", color="steelblue")
ax.plot(future_dates, fc_vals, label="Forecast", color="orange", marker="o")
ax.fill_between(future_dates, fc_lo, fc_hi, color="orange", alpha=0.15, label="95% CI")
ax.set_title(f"{TICKER} — Prophet 5-Day Ahead Forecast")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Run Prophet on All Stocks

In [ ]:
all_preds, all_fc, all_metrics = run_prophet_pipeline(processed, n_forecast=5)

metrics_df = pd.DataFrame(all_metrics)
print("\n── Prophet Results Across All Stocks ──")
print(metrics_df.to_string())
print(f"\nAvg MAPE: {metrics_df['MAPE'].mean():.2f}%")
print(f"Avg RMSE: {metrics_df['RMSE'].mean():.2f}")
print(f"Avg Dir Accuracy: {metrics_df['DirAcc'].mean():.1f}%")

In [ ]:
# Visualise all forecasts
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()
for i, (ticker, pred) in enumerate(all_preds.items()):
    if i >= 9: break
    ax = axes[i]
    actual = processed[ticker]["test"]
    ax.plot(actual, label="Actual", color="green", linewidth=1.5)
    ax.plot(pred, label="Prophet", color="orange", linewidth=1, linestyle="--")
    ax.set_title(STOCK_UNIVERSE.get(ticker, ticker), fontsize=10)
    ax.tick_params(axis="x", rotation=30, labelsize=7)
    ax.legend(fontsize=7)
plt.suptitle("Prophet Forecast vs Actual — All Stocks", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()